# Which component actually drives the detection

Reviewer 3 asks for the ICV components to be isolated: schema validation, duplicate
detection, numerical consistency, and their combinations. The paper reports Signals B
and C and their union, which hides the fact that Signal C is itself two tests.

This notebook takes the three primitives apart and scores every non-empty combination
on the 200 trajectories, against the same reference annotation. The signal code is
fetched from the released repository rather than reimplemented, and the published
union is asserted before anything else is reported.

In [ ]:
import ast, itertools, json, math, urllib.request
import numpy as np

BASE = ("https://raw.githubusercontent.com/achenna-source/"
        "icv-agent-observability/neucom-d-26-17269-r1/")

def fetch(p):
    return urllib.request.urlopen(BASE + p, timeout=90).read().decode("utf-8")

rows = [json.loads(l) for l in fetch("data/trajectories_200.jsonl").splitlines() if l.strip()]
steps = lambda r: ast.literal_eval(r["trajectory"]) if isinstance(r["trajectory"], str) else r["trajectory"]
TRAJ = [steps(r) for r in rows]
HUMAN = [r["label_human"] for r in rows]
FAIL = [h != "S" for h in HUMAN]
NF, NS = sum(FAIL), len(FAIL) - sum(FAIL)

ns = {}
exec(compile(fetch("code/signals.py"), "signals.py", "exec"), ns)
tier1, tier1_exempt, tier2 = ns["tier1"], ns["tier1_exempt"], ns["tier2"]
print("%d trajectories, %d failures, %d successes" % (len(TRAJ), NF, NS))

In [ ]:
from sentence_transformers import SentenceTransformer
enc = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")

def texts_of(tr):
    return [f"{s.get('thought','')} {s.get('action','')} {s.get('input','')}".strip() for s in tr]

MAXSIM = []
for tr in TRAJ:
    if len(tr) < 2:
        MAXSIM.append(0.0); continue
    e = enc.encode(texts_of(tr), normalize_embeddings=True)
    g = e @ e.T
    np.fill_diagonal(g, -1.0)
    MAXSIM.append(float(g.max()))
MAXSIM = np.array(MAXSIM)

# the three primitives, plus the repaired schema check
SCHEMA  = [tier1(tr)[0] for tr in TRAJ]
SCHEMA_X = [tier1_exempt(tr)[0] for tr in TRAJ]
DUP     = list(MAXSIM > 0.92)
NUM     = [tier2(tr, encoder=None)[2] for tr in TRAJ]
print("flagged by each primitive: schema %d, schema-exempt %d, duplicate %d, numeric %d"
      % (sum(SCHEMA), sum(SCHEMA_X), sum(DUP), sum(NUM)))

In [ ]:
def mcc(tp, fp, fn, tn):
    d = math.sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))
    return (tp*tn - fp*fn) / d if d else 0.0

def score(flags):
    tp = sum(1 for f, k in zip(flags, FAIL) if f and k)
    fp = sum(1 for f, k in zip(flags, FAIL) if f and not k)
    fn, tn = NF - tp, NS - fp
    prec = tp/(tp+fp) if tp+fp else float("nan")
    rec = tp/NF
    f1 = 2*prec*rec/(prec+rec) if prec+rec and prec == prec else float("nan")
    return tp, fp, prec, rec, f1, mcc(tp, fp, fn, tn)

def union(*flagsets):
    return [any(f) for f in zip(*flagsets)]

# sanity: the published union must reproduce before anything else is believed
pub = union(SCHEMA, DUP, NUM)
tp, fp, prec, rec, f1, m = score(pub)
assert (tp, fp) == (68, 25) and round(prec,3) == 0.731 and round(rec,3) == 0.531, \
    "published union does not reproduce: tp=%d fp=%d prec=%.3f rec=%.3f" % (tp, fp, prec, rec)
print("CHECK: B or C reproduces the paper, tp 68, fp 25, precision 0.731, coverage 0.531")

In [ ]:
PRIM = {"schema": SCHEMA, "duplicate": DUP, "numeric": NUM}

print("%-34s %4s %4s %7s %7s %7s %8s" % ("combination","tp","fp","prec","cov","F1","MCC"))
print("-"*78)
ROWS = []
for k in (1,2,3):
    for combo in itertools.combinations(PRIM, k):
        flags = union(*[PRIM[c] for c in combo])
        tp, fp, prec, rec, f1, m = score(flags)
        ROWS.append((" + ".join(combo), tp, fp, prec, rec, f1, m))
        print("%-34s %4d %4d %7.3f %7.3f %7.3f %+8.3f"
              % (" + ".join(combo), tp, fp, prec, rec, f1, m))

print()
print("with the recovered-detour exemption applied to the schema check:")
for name, flags in (("schema-exempt", SCHEMA_X),
                    ("schema-exempt + numeric", union(SCHEMA_X, NUM)),
                    ("schema-exempt + duplicate + numeric", union(SCHEMA_X, DUP, NUM))):
    tp, fp, prec, rec, f1, m = score(flags)
    print("%-34s %4d %4d %7.3f %7.3f %7.3f %+8.3f" % (name, tp, fp, prec, rec, f1, m))

In [ ]:
# what each primitive adds on top of the other two, which is the question that matters
print("marginal contribution, failures caught that the other two miss:")
for name in PRIM:
    others = [PRIM[c] for c in PRIM if c != name]
    rest = union(*others)
    uniq = sum(1 for f, r, k in zip(PRIM[name], rest, FAIL) if f and not r and k)
    uniq_fp = sum(1 for f, r, k in zip(PRIM[name], rest, FAIL) if f and not r and not k)
    print("   %-10s adds %2d true positives and %2d false alarms" % (name, uniq, uniq_fp))

print()
print("overlap between duplicate and numeric:")
both = sum(1 for d, n in zip(DUP, NUM) if d and n)
print("   trajectories flagged by duplicate: %d, of which also by numeric: %d" % (sum(DUP), both))

In [ ]:
# per failure mode, which primitive reaches it
modes = sorted({h for h in HUMAN if h != "S"})
print("%-6s %6s %9s %11s %9s %9s" % ("mode","n","schema","duplicate","numeric","union"))
for m_ in modes:
    idx = [i for i, h in enumerate(HUMAN) if h == m_]
    row = [sum(1 for i in idx if f[i]) for f in (SCHEMA, DUP, NUM, union(SCHEMA, DUP, NUM))]
    print("%-6s %6d %9d %11d %9d %9d" % (m_, len(idx), *row))

In [ ]:
print("=== for the manuscript ===")
for name, tp, fp, prec, rec, f1, m in ROWS:
    print("%-24s tp %2d, fp %2d, precision %.3f, coverage %.3f, MCC %+.3f"
          % (name, tp, fp, prec, rec, m))